In [1]:
project_root_dir = !git rev-parse --show-toplevel
project_root_dir = project_root_dir[0]

import os
import sys
import json

import pika

sys.path.append(os.path.join(project_root_dir, 'lib'))
from autoincrement import *

In [ ]:
server = AutoincrementServer('/home/misha/tmp/autoincs.json', RMQ_DEFAULT_CONNECTION_URL, verbosity=2)
server.run()

Loaded 3 keys


In [4]:
# class AutoincrementServer:
#     def __init__(self, rmq_connection_url, storage_fname, verbosity=0):
#         self.connection_parameters = pika.URLParameters(rmq_connection_url)
#         self.connection = pika.BlockingConnection(self.connection_parameters)
#         self.channel = self.connection.channel()
#         queue = self.channel.queue_declare(
#             queue=RMQ_AUTOINCREMENT_REQUESTS_QUEUE_NAME,
#             durable=True,
#             arguments={'x-single-active-consumer': True},
#         )
#         self.autoincs = defaultdict(int)
#         self.verbosity = verbosity
#         self.storage_fname = storage_fname

#         if os.path.exists(storage_fname):
#             if_verbose(self.verbosity, 3, lambda: print(f'Loading autoincrements from "{storage_fname}"'))
            
#             with open(storage_fname, 'r') as f:
#                 loaded = json.load(f)
#                 self.autoincs.update(loaded)

#                 if self.verbosity >= 3:
#                     for k, v in self.autoincs.items():
#                         print(f'Loaded {k}={v}')
                
#                 if_verbose(self.verbosity, 1, lambda: print(f'Loaded {len(self.autoincs)} keys'))

#     def run(self):
#         self.channel.basic_qos(prefetch_count=1) # max 1 unacked message, i.e. serial processing
#         self.channel.basic_consume(
#             queue=RMQ_AUTOINCREMENT_REQUESTS_QUEUE_NAME, 
#             on_message_callback=self.on_request, 
#             auto_ack=False)
#         self.channel.start_consuming()

#     def on_request(self, ch, method, properties, body):
#         if_verbose(self.verbosity, 3, lambda: print(f'on_message: method={method}, properties={properties}, len(body)={len(body)}'))
#         reply_to = properties.reply_to

#         key = body.decode()
#         if_verbose(self.verbosity, 2, lambda: print(f'Generating autoincrement for {key=}'))
#         self.autoincs[key] += 1
#         if_verbose(self.verbosity, 2, lambda: print(f'Generated autoincrement for {key=} is {self.autoincs[key]}'))
        
#         with open(self.storage_fname, 'w') as f:
#             json.dump(self.autoincs, f)
#             if_verbose(self.verbosity, 1, lambda: print(f'Saved {len(self.autoincs)} keys to "{self.storage_fname}"'))

#         properties = pika.spec.BasicProperties(
#             delivery_mode=pika.DeliveryMode.Persistent
#         )
#         self.channel.basic_publish(
#             exchange='', 
#             routing_key=reply_to, 
#             body=str(self.autoincs[key]).encode(),
#             properties=properties)
#         ch.basic_ack(delivery_tag=method.delivery_tag)